# 2026년 뷰티 트렌드 → 아누아 심층 리서치
기사 수집 → 표현 탐색 → 개념별 비중 비교 → 관련 브랜드 조사 순서입니다.
현재 자료는 수집기 검증용 소량 표본입니다. 트렌드 결론은 아직 없습니다. 자세한 기준은 README.md를 읽어주세요.

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'pipeline.py').exists():
    ROOT = ROOT / 'projects' / 'beauty-trend-analysis'
assert (ROOT / 'pipeline.py').exists(), '이 노트북이 있는 폴더에서 실행하세요.'
PYTHON = ROOT / '.venv' / 'Scripts' / 'python.exe'
if not PYTHON.exists():
    PYTHON = Path(sys.executable)
def run(*args):
    result = subprocess.run([str(PYTHON), '-X', 'utf8', str(ROOT / 'pipeline.py'), *args], cwd=ROOT, capture_output=True, text=True, encoding='utf-8')
    print(result.stdout)
    if result.returncode:
        raise RuntimeError(result.stderr)
print(ROOT)

## 1. 수집
네트워크를 사용합니다. robots 확인 및 최소 2초 간격을 적용합니다. 동일 URL은 재저장하지 않습니다. 기간 필터가 과거 전체 수집을 보장하지 않습니다.

In [ ]:
# 새 자료를 수집할 때만 주석을 해제하세요.
# run('crawl', '--pages', '2', '--limit', '20', '--start', '2025-01-01', '--end', '2026-10-11')
articles = pd.read_json(ROOT / 'data/articles.jsonl', lines=True)
articles[['source', 'role', 'published_at', 'title', 'url']]

## 2. 텍스트 분석
TF-IDF 탐색과 유사어 사전 기반 비중 분석을 함께 실행합니다. 화해의 선택 리포트는 통계 분모에서 제외합니다.

In [ ]:
run('analyze', '--year', '2026', '--cutoff', '2026-10-11')
pd.read_csv(ROOT / 'outputs/coverage.csv')

In [ ]:
# TF-IDF가 높은 표현은 검토 후보이지 확정 트렌드가 아닙니다.
pd.read_csv(ROOT / 'outputs/discovered_terms.csv').head(30)

In [ ]:
# 비교 기간 자료가 없으면 변화량은 NaN입니다.
pd.read_csv(ROOT / 'outputs/period_comparison.csv').sort_values('change_pp', ascending=False)

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'outputs/keyword_share.png')))

## 3. 근거 검토와 아누아 심층 조사
근거 문맥을 확인한 뒤 키워드를 선정합니다. 관련 아누아 기사가 0건이면 자료를 추가로 확보해야 합니다. 제품·리뷰 자동 수집은 아직 없습니다.

In [ ]:
pd.read_csv(ROOT / 'outputs/keyword_evidence.csv').head(20)

In [ ]:
pd.read_csv(ROOT / 'outputs/brand_drilldown.csv')

## 4. 후속 리뷰 분석
실제 리뷰를 확보한 후 `review-code --input data/reviews.jsonl`을 실행합니다. 필수 형식은 README를 참고하세요. 속성별 감성은 문맥을 읽고 수동 코딩합니다. 리뷰를 만들어 넣거나 기사 본문을 리뷰로 대체하지 않습니다.